# aoi-sentinel — Colab Quickstart

Mamba-RL false-call filter for SMT AOI on a free Colab T4 (or Pro L4/A100).

## What you'll run
1. GPU sanity check + clone repo
2. Install deps (torch + mamba-ssm + project) — robust to Colab torch updates
3. Download VisA PCB benchmark data
4. **Stage 0** — supervised pretraining of MambaVision on VisA (~30–60 min on T4)
5. **Stage 1** — Lagrangian PPO on the NPI simulator (~30–60 min)
6. Plot cost / escape / λ trajectories

**Runtime → Change runtime type → GPU** before running.

Tip: free Colab disconnects after ~12h idle. Mount Google Drive and point checkpoint paths there if you want persistence.

## 1. GPU + repo

In [ ]:
!nvidia-smi | head -16

In [ ]:
import os
if not os.path.exists('/content/aoi-sentinel'):
    !git clone https://github.com/DrJinHoChoi/aoi-sentinel.git /content/aoi-sentinel
%cd /content/aoi-sentinel
!git pull

## 2. Mamba kernel install — SKIPPED on Colab (use pure-PyTorch fallback)

`mamba-ssm` ships custom CUDA kernels that **consistently fail to build on
Colab's current Python 3.12 + torch 2.10 combination** — even with
`MAMBA_SKIP_CUDA_BUILD=TRUE`. There's no clean install path.

That's fine. We have a pure-PyTorch fallback at
`aoi_sentinel/models/vmamba/pure_torch_mamba.py` that runs the same
selective-scan math on any torch installation. It's ~5-15× slower but
**no install required**.

For full kernel speed, switch to WSL2 (see [docs/setup_wsl.md](https://github.com/DrJinHoChoi/aoi-sentinel/blob/main/docs/setup_wsl.md)) or DGX Spark.

In [ ]:
# Quick env diagnostic — useful for debugging if something goes wrong later.
import torch, sys
print(f'python : {sys.version_info.major}.{sys.version_info.minor}')
print(f'torch  : {torch.__version__}')
print(f'cuda   : {torch.version.cuda}')
print(f'device : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print()
print('Mamba install: SKIPPED — using pure-PyTorch fallback (see cell above).')

In [ ]:
# Project + the rest. Torch is already in Colab; we install everything else here.
# Install order matters: mambavision FIRST so its dependency resolver picks the
# right timm version, then the rest without pinning timm.
%cd /content/aoi-sentinel
!pip install -q --ignore-requires-python -e "." --no-deps
!pip install -q mambavision
!pip install -q albumentations gymnasium tensorboard wandb \
    rich click pyyaml lxml pyarrow pandas opencv-python-headless \
    scikit-learn scikit-image fastapi uvicorn jinja2 python-multipart


In [ ]:
# Sanity check — both encoders auto-resolve to whatever's available.
# - ImageEncoder tries MambaVision (native API → timm registration → ConvNeXt fallback)
# - get_mamba_block tries mamba-ssm CUDA kernel → pure-PyTorch fallback
# Either way the (B, embed_dim) and (B, L, d_model) contracts hold so the rest
# of the pipeline (RL, classifier, eval) is oblivious.
import torch
from aoi_sentinel.models.vmamba.image_encoder import ImageEncoder
from aoi_sentinel.models.vmamba.pure_torch_mamba import get_mamba_block

img_enc = ImageEncoder(size='tiny', pretrained=True).cuda().eval()
x = torch.randn(2, 3, 224, 224).cuda()
with torch.no_grad():
    feat = img_enc(x)
print(f'vision out: {tuple(feat.shape)}   backend: {img_enc.backend_used}')

mb = get_mamba_block(d_model=128).cuda().eval()
y = torch.randn(2, 64, 128).cuda()
with torch.no_grad():
    seq = mb(y)
print(f'seq out:    {tuple(seq.shape)}')

print('OK — encoders ready.')

## 3. Download benchmark data

VisA full archive is ~4.5 GB. We grab it once and prune to the 4 PCB classes (~1.2 GB).

**Optional**: mount Google Drive and copy `data/raw/visa/` there to skip re-downloading on future sessions.

In [ ]:
# Optional Drive mount — uncomment if you want checkpoint/data persistence.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!python scripts/download_visa.py --out data/raw/visa --pcb-only

In [ ]:
!ls data/raw/visa/ && echo '---' && ls data/raw/visa/pcb1/Data/Images/ 2>/dev/null | head

## 4. Stage 0 — pretrain MambaVision on VisA

Cost-sensitive supervised classifier. Output: image-encoder weights for the RL stage.

In [ ]:
!aoi train pretrain --config configs/stage0_pretrain_colab.yaml

## 5. Stage 1 — Mamba RL on the NPI simulator

Lagrangian PPO under the escape-rate budget. Watch for:
- `avg_cost` decreasing over iterations
- `λ` rising when escape budget is violated, falling when satisfied
- `cum_escape` should stay near 0

In [ ]:
!aoi train npi-rl --config configs/stage1_npi_rl_colab.yaml 2>&1 | tee /content/stage1.log

## 6. Plot trajectories

In [ ]:
import re
import matplotlib.pyplot as plt

rows = []
pat = re.compile(r'iter (\d+).*?λ=([\-\d\.]+).*?avg_cost=([\-\d\.]+).*?cum_cost=([\-\d\.]+).*?cum_escape=(\d+)')
with open('/content/stage1.log') as f:
    for line in f:
        m = pat.search(line)
        if m:
            rows.append([int(m[1]), float(m[2]), float(m[3]), float(m[4]), int(m[5])])

import numpy as np
rows = np.array(rows)
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, col, name in zip(axes, [1, 2, 3, 4], ['λ (Lagrange)', 'avg cost / step', 'cumulative cost', 'cumulative escapes']):
    ax.plot(rows[:, 0], rows[:, col])
    ax.set_xlabel('iter'); ax.set_title(name); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Next

- Run with `--config configs/stage1_npi_rl.yaml` (full size) on a Colab Pro A100 / L4 for the proper baseline numbers.
- Add DeepPCB and SolDef_AI to the data mix (`configs/.../data.dataset`).
- When real Saki data arrives, swap `aoi_sentinel.data.benchmarks` for `aoi_sentinel.data.saki` and rerun stage 1 with `init_from` pointing at the stage 0 checkpoint.